In [2]:
import json # Let's pandas work with JSON files.


import pandas as pd 

In [3]:
# Opens the raw ATT&CK JSON file for reading 

with open("../data/raw/enterprise-attack.json", "r") as file:
    attack_data = json.load(file) # This step converts the JSON contents into Python objects

In [4]:
# Shows what Python data type the loaded JSON became

type(attack_data)

dict

In [5]:
# Shows the top-level keys available in the ATT&CK JSON bundle

attack_data.keys()

dict_keys(['type', 'id', 'objects'])

In [7]:
# Counts the total number of STIX objects in the ATT&CK dataset

len(attack_data["objects"]) 

26086

In [8]:
# Counter helps count how many times each object type appears 

from collections import Counter

# Reads the "type" field from every stix object 

object_types = Counter(obj.get("type") for obj in attack_data["objects"])

#Displays the count of each object type 

object_types 

Counter({'relationship': 21262,
         'x-mitre-analytic': 1758,
         'attack-pattern': 858,
         'malware': 733,
         'x-mitre-detection-strategy': 699,
         'course-of-action': 268,
         'intrusion-set': 191,
         'x-mitre-data-component': 109,
         'tool': 95,
         'campaign': 56,
         'x-mitre-data-source': 38,
         'x-mitre-tactic': 15,
         'x-mitre-collection': 1,
         'x-mitre-matrix': 1,
         'identity': 1,
         'marking-definition': 1})

In [9]:
# Keeps only ATT&CK technique/sub-technique objects 

techniques = [obj for obj in attack_data["objects"] if obj.get("type") == "attack-pattern"]

# Confirms how many attack pattern objects were extracted

len(techniques) 


858

In [10]:
# Displays the first ATT&CK technique object so you can inspect its raw STIX structure 

techniques[0]

{'type': 'attack-pattern',
 'spec_version': '2.1',
 'id': 'attack-pattern--0042a9f5-f053-4769-b3ef-9ad018dfa298',
 'created': '2020-01-14T17:18:32.126Z',
 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5',
 'revoked': False,
 'external_references': [{'source_name': 'mitre-attack',
   'url': 'https://attack.mitre.org/techniques/T1055/011',
   'external_id': 'T1055.011'},
  {'source_name': 'Elastic Process Injection July 2017',
   'description': 'Hosseini, A. (2017, July 18). Ten Process Injection Techniques: A Technical Survey Of Common And Trending Process Injection Techniques. Retrieved December 7, 2017.',
   'url': 'https://www.endgame.com/blog/technical-blog/ten-process-injection-techniques-technical-survey-common-and-trending-process'},
  {'source_name': 'MalwareTech Power Loader Aug 2013',
   'description': 'MalwareTech. (2013, August 13). PowerLoader Injection – Something truly amazing. Retrieved December 16, 2017.',
   'url': 'https://www.malwaretech.com/2013/08

In [11]:
# Shows only the field names available in the first technique object 

techniques[0].keys()

dict_keys(['type', 'spec_version', 'id', 'created', 'created_by_ref', 'revoked', 'external_references', 'object_marking_refs', 'modified', 'name', 'description', 'kill_chain_phases', 'x_mitre_attack_spec_version', 'x_mitre_deprecated', 'x_mitre_domains', 'x_mitre_is_subtechnique', 'x_mitre_modified_by_ref', 'x_mitre_platforms', 'x_mitre_version'])

In [27]:
# Creates an empty list that will hold one cleaned record per ATT&CK technique 

records = [] 


for technique in techniques:
    attack_id = None # Default value in case an ATT&CK ID is not found
    
    
    for reference in technique.get("external_references", []):
        if reference.get("source_name") == "mitre-attack":
            attack_id = reference.get("external_id") # Extracts IDs such as T1059 or T1059.001
            break
            
    tactics = [
        phase.get("phase_name")
        for phase in technique.get("kill_chain_phases", [])
    ]  # Extracts tactic names such as execution or credential-access
    
    records.append({
        "technique_id": attack_id, 
        "technique_name": technique.get("name"),
        "description": technique.get("description"), 
        "tactics": tactics, 
        "platforms": technique.get("x_mitre_platforms", []),
        "is_subtechnique": technique.get("x_mitre_is_subtechnique", False),
        "created": technique.get("created"),
        "modified": technique.get("modified"),
        "deprecated": technique.get("x_mitre_deprecated", False),
        "revoked": technique.get("revoked", False)
        
    })  # Adds one cleaned technique record to our list 


In [28]:
df_techniques = pd.DataFrame(records)  # Converts the list of dictionaries into a tabular dataframe

In [29]:
df_techniques.head()  # Displays the first 5 cleaned ATT&CK techniques

,technique_id,technique_name,description,tactics,platforms,is_subtechnique,created,modified,deprecated,revoked
0,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into pro...,"[stealth, privilege-escalation]",[Windows],True,2020-01-14T17:18:32.126Z,2026-05-12T15:12:00.617Z,False,False
1,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,"[execution, persistence, privilege-escalation]",[Windows],True,2019-11-27T14:58:00.429Z,2026-05-12T15:12:00.618Z,False,False
2,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,"[stealth, persistence, command-and-control]","[Linux, macOS, Windows]",True,2022-09-30T21:18:41.930Z,2026-05-12T15:12:00.619Z,False,False
3,T1066,Indicator Removal from Tools,If a malicious tool is detected and quarantine...,[stealth],"[Linux, macOS, Windows]",False,2017-05-31T21:30:54.176Z,2026-04-14T22:53:15.802Z,False,True
4,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,[collection],"[Linux, macOS, Windows]",True,2020-02-20T21:01:25.428Z,2026-05-12T15:12:00.619Z,False,False


In [30]:
df_techniques.shape  # Shows the number of rows and columns in the dataframe

(858, 10)

In [31]:
df_techniques.isnull().sum() # Counts missing values in each column


technique_id       0
technique_name     0
description        0
tactics            0
platforms          0
is_subtechnique    0
created            0
modified           0
deprecated         0
revoked            0
dtype: int64

In [32]:
df_techniques["technique_id"].duplicated().sum() # Counts duplicate ATT&CK technique IDs

0

In [33]:
df_techniques["revoked"].value_counts() # Shows how many techniques are active vs revoked

revoked
False    709
True     149
Name: count, dtype: int64

In [34]:
df_techniques["deprecated"].value_counts() # Shows how many techniques are active vs deprecated

deprecated
False    846
True      12
Name: count, dtype: int64

In [35]:
df_techniques[df_techniques["technique_id"].isnull()] # Displays any rows where the ATT&CK ID could not be extracted

,technique_id,technique_name,description,tactics,platforms,is_subtechnique,created,modified,deprecated,revoked


In [36]:
df_active_techniques = df_techniques[(df_techniques["revoked"] == False) & (df_techniques["deprecated"] == False)].copy() # Keeps only techniques that are neither revoked nor deprecated

In [37]:
df_active_techniques.shape  # Shows how many active techniques remain after filtering

(697, 10)

In [38]:
df_active_techniques["revoked"].value_counts()  # Confirms all remaining rows are not revoked

revoked
False    697
Name: count, dtype: int64

In [39]:
df_active_techniques["deprecated"].value_counts()  # Confirms all remaining rows are not deprecated

deprecated
False    697
Name: count, dtype: int64

In [40]:
df_techniques.to_csv(

    "../data/processed/mitre_techniques_all.csv",

    index=False

)  # Saves the full 858-row ATT&CK technique dataset, including revoked and deprecated records

In [41]:
df_active_techniques.to_csv(

    "../data/processed/mitre_techniques_active.csv",

    index=False

)  # Saves only the 697 active techniques for analysis

In [42]:
import os  # Gives Python access to operating-system file utilities

os.listdir("../data/processed")  # Lists the files currently stored in data/processed/

['.gitkeep', 'mitre_techniques_all.csv', 'mitre_techniques_active.csv']

In [43]:
pd.read_csv("../data/processed/mitre_techniques_all.csv").shape  # Confirms the full exported CSV has the expected dimensions

(858, 10)

In [44]:
pd.read_csv("../data/processed/mitre_techniques_active.csv").shape  # Confirms the active exported CSV has the expected dimensions

(697, 10)

In [45]:
df_technique_tactics = (df_active_techniques[
    ["technique_id", "technique_name", "tactics"]].explode("tactics") # Creates one row per technique-tactic relationship
    .rename(columns={"tactics": "tactic"}) # Renames the exploded column to a sungular name 
    .reset_index(drop=True) # Rebuilds the dataframe index after exploding rows
                       )







In [46]:
df_technique_tactics.head(10)  # Shows the first 10 technique-to-tactic mappin

,technique_id,technique_name,tactic
0,T1055.011,Extra Window Memory Injection,stealth
1,T1055.011,Extra Window Memory Injection,privilege-escalation
2,T1053.005,Scheduled Task,execution
3,T1053.005,Scheduled Task,persistence
4,T1053.005,Scheduled Task,privilege-escalation
5,T1205.002,Socket Filters,stealth
6,T1205.002,Socket Filters,persistence
7,T1205.002,Socket Filters,command-and-control
8,T1560.001,Archive via Utility,collection
9,T1021.005,VNC,lateral-movement


In [47]:
df_technique_tactics.shape  # Shows how many technique-tactic relationships exist

(872, 3)

In [48]:
df_technique_platforms = (

    df_active_techniques[

        ["technique_id", "technique_name", "platforms"]

    ]

    .explode("platforms")  # Creates one row per technique-platform relationship

    .rename(columns={"platforms": "platform"})  # Renames the exploded column to a singular name

    .reset_index(drop=True)  # Rebuilds the dataframe index after exploding rows

)

In [49]:
df_technique_platforms.head(10)  # Shows the first 10 technique-to-platform mappings

,technique_id,technique_name,platform
0,T1055.011,Extra Window Memory Injection,Windows
1,T1053.005,Scheduled Task,Windows
2,T1205.002,Socket Filters,Linux
3,T1205.002,Socket Filters,macOS
4,T1205.002,Socket Filters,Windows
5,T1560.001,Archive via Utility,Linux
6,T1560.001,Archive via Utility,macOS
7,T1560.001,Archive via Utility,Windows
8,T1021.005,VNC,Linux
9,T1021.005,VNC,Windows


In [50]:
df_technique_platforms.shape  # Shows how many technique-platform relationships exist

(1846, 3)

In [51]:
df_technique_tactics["tactic"].isnull().sum()  # Counts techniques that ended up without a tactic

0

In [52]:
df_technique_platforms["platform"].isnull().sum()  # Counts techniques that ended up without a platform

0

In [53]:
sorted(df_technique_tactics["tactic"].dropna().unique())  # Lists every unique ATT&CK tactic in alphabetical order

['collection',
 'command-and-control',
 'credential-access',
 'defense-impairment',
 'discovery',
 'execution',
 'exfiltration',
 'impact',
 'initial-access',
 'lateral-movement',
 'persistence',
 'privilege-escalation',
 'reconnaissance',
 'resource-development',
 'stealth']

In [54]:
sorted(df_technique_platforms["platform"].dropna().unique())  # Lists every unique platform in alphabetical order

['Containers',
 'ESXi',
 'IaaS',
 'Identity Provider',
 'Linux',
 'Network Devices',
 'Office Suite',
 'PRE',
 'SaaS',
 'Windows',
 'macOS']

In [55]:
df_technique_tactics.to_csv(
    "../data/processed/mitre_technique_tactics.csv",
    index=False
)  # Saves one row per technique-to-tactic relationship

In [56]:
df_technique_platforms.to_csv(

    "../data/processed/mitre_technique_platforms.csv",

    index=False

)  # Saves one row per technique-to-platform relationship

In [57]:
os.listdir("../data/processed")  # Lists every file currently stored in the processed-data folder

['.gitkeep',
 'mitre_techniques_all.csv',
 'mitre_techniques_active.csv',
 'mitre_technique_tactics.csv',
 'mitre_technique_platforms.csv']

In [58]:
pd.read_csv("../data/processed/mitre_technique_tactics.csv").shape  # Confirms the tactic mapping CSV was saved correctly

(872, 3)

In [59]:
pd.read_csv("../data/processed/mitre_technique_platforms.csv").shape  # Confirms the platform mapping CSV was saved correctly

(1846, 3)